# STAMMP Batched Pipeline — Full Walkthrough

**STAMMP** (Sequence To Alignment and Metabolic Model Pipeline) processes paired-end metagenomic FASTQ files through human depletion, taxonomic classification, AGORA genome mapping, and MICOM community metabolic modeling.

This notebook is a **reference guide only** — all cells are markdown or raw text and cannot be executed. Copy commands into your terminal.

---

## Pipeline overview

The pipeline runs in two major stages, each requiring its own conda environment:

**Stage 1 — AGORA (Snakemake, `ut-agora_pipeline_clone` env)**
1. `qc_reads` — adapter trimming and quality filtering with fastp
2. `bowtie_filter` — human read depletion (CHM13+Y reference)
3. `kraken_classify` — taxonomic classification with Kraken2
4. `kraken_extract` — discard eukaryotic reads (taxid 2759), compress survivors
5. `minimap_agora_count` — align filtered reads to AGORA genomes, count hits per taxon

After all batches complete, per-sample `.counts` files are merged into a single CSV.

**Stage 2 — MICOM (`micomPy310` env)**
1. Normalize read counts by genome size
2. Build community metabolic models from AGORA species database
3. Simulate community growth on a Western diet medium (tradeoff = 0.7)
4. Save growth results

Batching is done upfront with `batch_samples.sh`, and the orchestration script `run_batched_snakemakes.sh` handles running each batch sequentially, merging results, and launching MICOM.

---
## 1. Conda environments

The pipeline uses two separate environments that cannot be combined. Activate the correct one before each stage.

**AGORA / Snakemake environment** — contains Snakemake, fastp, Bowtie2, Kraken2, minimap2, samtools, pigz, and the agora filter scripts dependencies.

**MICOM environment** — contains Python 3.10, micom, cobra, and pandas. AGORA tools are not available here.

---
## 2. Data preparation

### 2a. Downloading samples from ENA

Sample accession lists are pulled from ENA using the notebook at:

`/home/maholton/mah_note/liver_disease_metanalysis/gettingListOfSamples.ipynb`

Download paired-end reads using `fastq-dump` or `fasterq-dump`. Place all downloaded `*_1.fastq.gz` / `*_2.fastq.gz` files in a single flat directory before batching.

### 2b. Setting up the project directory

Create a project directory and copy the STAMMP pipeline into it. The pipeline can be shared by zipping the STAMMP directory — no install step is needed.

### 2c. Batching samples with `batch_samples.sh`

The Snakemake workflow processes one batch directory at a time. `batch_samples.sh` splits a flat directory of FASTQ pairs into numbered `batch_01/`, `batch_02/`, ... subdirectories. This keeps memory and intermediate file counts manageable.

**Usage:**

After running, your raw data directory will look like:

```
raw_fastqs/
├── batch_01/
│   ├── SRR123456_1.fastq.gz
│   ├── SRR123456_2.fastq.gz
│   ├── SRR123457_1.fastq.gz
│   └── SRR123457_2.fastq.gz
├── batch_02/
│   └── ...
```

The `--input` argument to `run_batched_snakemakes.sh` points to this parent directory (not an individual batch).

---
## 3. Running the full pipeline

### 3a. Quick start

With the AGORA conda environment active, run the orchestration script. It handles all batches, the count merge, and MICOM automatically.

### 3b. Full argument reference for `run_batched_snakemakes.sh`

| Argument | Required | Default | Description |
|---|---|---|---|
| `--input <dir>` | Yes | — | Parent directory containing `batch_01/`, `batch_02/`, ... subdirectories |
| `--output <dir>` | Yes | — | Project output root; all pipeline outputs go under `<dir>/outputs/` |
| `--cores <N>` | No | 32 | Total cores passed to Snakemake. Snakemake schedules jobs so concurrent threads stay ≤ this value |
| `--skip-micom` | No | off | If set, stop after the AGORA merge step and do not run MICOM |

**Examples:**

### 3c. Overriding cluster database paths

The Snakemake file defaults to paths on the Zarrinpar cluster. If you are running on a different system, override the database paths by passing them through to Snakemake. You can do this by temporarily editing `run_batched_snakemakes.sh`, or by calling Snakemake directly on a single batch (see section 4).

The three database paths are:

| Config key | Default path | Description |
|---|---|---|
| `kraken_db` | `/mnt/zarrinpar/Pynchon/Databases/kraken2/k2_standard_08_GB_20250714` | Kraken2 database directory |
| `bowtie_idx` | `/mnt/zarrinpar/Pynchon/Databases/human_genome/bowtie_CHM13plusY/...` | Bowtie2 index prefix (CHM13+Y) |
| `agora_db` | `/mnt/zarrinpar/scratch/rrichter/.../agora_genomes.idx` | minimap2 AGORA genome index |

---
## 4. Step-by-step walkthrough (running stages manually)

The orchestration script handles everything in sequence, but you can also run each stage individually. This is useful for debugging, resuming a partial run, or targeting a specific Snakemake rule.

### 4a. Batching and sample list generation

`run_batched_snakemakes.sh` auto-generates the sample list from all `batch_*/` directories on startup. If running manually, create it yourself:

### 4b. Running a single batch through the AGORA Snakemake

Activate the AGORA environment and call Snakemake directly on one batch. `--rerun-incomplete` ensures that any interrupted jobs are restarted cleanly.

Repeat for `batch_02`, `batch_03`, etc., changing only `raw_dir`. Snakemake is idempotent — it skips any sample that already has a `.counts` file in `AGORA_mapped/`.

**Parallelism note:** The `--cores` argument is the total thread budget. `threads` (per-rule allocation) controls how many samples run concurrently: `concurrent_samples = floor(--cores / threads)`. With `--cores 32` and `threads=8`, four samples run in parallel. Reducing `threads` increases parallelism; increasing it (up to `--cores`) serializes.

### 4c. Running a specific rule with `--until`

Stop the pipeline at a specific rule — useful to inspect intermediate files.

### 4d. Merging per-sample count files

After all batches have finished, combine every `.counts` file into a single CSV. `count_files_to_table.py` reads each tab-separated `taxon_id\tcount` file and pivots them into a matrix (rows = taxon IDs, columns = sample IDs).

### 4e. Running MICOM

Switch to the MICOM environment. `MICOM_pipeline.py` must be called with its full path or from the STAMMP root directory so it can locate `utils/analysis.py`.

**MICOM arguments:**

| Argument | Description |
|---|---|
| `--agora` / `-a` | Path to `agora_mapped_counts.csv` produced by the merge step |
| `--output` / `-o` | Project outputs root (`micom_outputs/` is created inside here) |
| `--reference` / `-r` | Path to the `micom_references/` directory inside the STAMMP install |

MICOM runs three sub-steps internally:
1. Normalize read counts by genome size (writes `abundance_normalized_table.csv` and `abundance_normalized.csv` to `micom_outputs/processed/`)
2. Build community metabolic models from `agora_species_103.qza`, cutoff = 0.0001 (writes per-sample model files to `micom_outputs/models/`)
3. Grow models on Western diet medium (`western_diet_gut_agora.qza`), tradeoff = 0.7, and save results to `micom_outputs/models/res_western.zip`

---
## 5. Output structure

After a complete run, the project output directory will contain:

```
<project_output>/
└── outputs/
    ├── agora_outputs/
    │   ├── sample_list.txt               — master list of all sample IDs
    │   ├── cleaned/                      — fastp-trimmed reads (*_1/2.fastq.gz)
    │   ├── bowtie_filtered/              — human-depleted reads (*_1/2.fastq)
    │   ├── kraken_filtered/              — eukaryote-filtered reads (*_1/2.fastq.gz)
    │   │                                   + Kraken2 output (*.kraken, *.report)
    │   └── AGORA_mapped/
    │       ├── <sample>.counts           — per-sample taxon hit counts (tab-separated)
    │       └── agora_mapped_counts.csv   — merged count matrix (taxon x sample)
    ├── micom_outputs/
    │   ├── processed/
    │   │   ├── abundance_normalized_table.csv
    │   │   └── abundance_normalized.csv
    │   └── models/
    │       ├── <sample>.pickle           — per-sample MICOM model files
    │       └── res_western.zip           — GrowthResults object (Western diet)
    └── humann_outputs/                   — reserved for HUMAnN (run separately)
```

Intermediate files (cleaned reads, Bowtie-filtered reads) are deleted by the `minimap_agora_count` rule after a sample's `.counts` file is produced, to conserve disk space.

---
## 6. Common issues and troubleshooting

**Job killed with SIGINT (signal 2) mid-run**
This usually indicates the job was interrupted (Ctrl+C, HPC time limit, or an OOM kill). Re-run with `--rerun-incomplete` — Snakemake detects the partial output and restarts only the affected samples.

**`ERROR: no batch_*/ directories found`**
`--input` should point to the parent directory containing `batch_01/`, `batch_02/`, etc. — not to an individual batch subdirectory.

**`ERROR: no .counts files found` after all batches finish**
Check that `AGORA_mapped/` exists under `--output/outputs/agora_outputs/`. If a batch failed silently, re-run it with `--rerun-incomplete` before attempting the merge.

**MICOM solver error or missing GLPK**
MICOM uses the default solver configured in `cobra`. Confirm the correct environment is active (`micomPy310`) and that `cobra.Configuration().solver` returns a working solver. The pipeline prints the default solver at startup.

**`ModuleNotFoundError: utils`**
`MICOM_pipeline.py` auto-detects the STAMMP root from its own path. If you moved or copied the script outside the STAMMP directory, the import will fail. Always call it via a full path from within the STAMMP root, or use the path provided in the run command above.